# MATLAB for Filter Coefficients

## What you'll learn in this module

- How MATLAB generates practical filter coefficients for FPGA DSP.
- How a 4th-order filter maps into two 2nd-order stages for RTL implementation.
- How to transfer and place generated coefficients into the SystemVerilog module correctly.

MATLAB is a calculation tool. For this workshop, we will use it to generate filter coefficients.

These coefficients are what you will copy into your SystemVerilog filter modules.

If you’ve never used MATLAB, follow the setup guide [here](matlab_setup.ipynb)

## How to Use MATLAB Scripts

We are using MATLAB’s built-in filter design tools to generate filter coefficients needed for your SystemVerilog filter.

These coefficients are copied directly into your `fourth_order_filter` module.

### What you need to do 
For this workshop, you only need to:
1. Change the cutoff frequency (`Fc`) or the passband frequencies (`F_pass`)
2. Run the script
3. Copy the generated coefficients into your SystemVerilog code

### Understanding the output 
MATLAB designs a 4th-order filter, but splits it into:
- Two 2nd-order stages
- That’s why you see two rows of coefficients

After running the script, MATLAB will print a table of values in this format:

```python
B0   B1   B2   A0   A1   A2
B0   B1   B2   A0   A1   A2
```

Important:
- Skip the 4th value (`A0`)

**Example:** 

MATLAB output:

![matlab_terminal.png](attachment:matlab_terminal.png)

Becomes:

```systemverilog
logic signed [15:0] filtered_audio;
logic filtered_valid;

fourth_order_filter #(
    // Stage 1
    .STAGE1_B0(100),
    .STAGE1_B1(200),
    .STAGE1_B2(100),
    .STAGE1_A1(-300),
    .STAGE1_A2(150),

    // Stage 2
    .STAGE2_B0(32768),
    .STAGE2_B1(-65536),
    .STAGE2_B2(32768),
    .STAGE2_A1(-64000),
    .STAGE2_A2(31000)
) u_low_pass_filter (
    .clk  (clk_50),
    .reset(reset),

    .x(in_audio),
    .x_valid(in_valid),

    .y(filtered_audio),
    .y_valid(filtered_valid)
  );

```

## Low-pass Filter

![lpf.png](attachment:lpf.png)

Copy and paste the following MATLAB code:
```ts
Fc = 1000;      % Cutoff frequency in Hz (CHANGE THIS)

% Leave these values unchanged
Fs = 48100;     % Sample rate
N  = 4;         % Filter order
Rp = 1;         % Passband ripple in dB
Rst = 60;       % Stopband attenuation in dB
scale = 2^15;   % Fixed-point scaling

% Design the low-pass elliptic IIR filter
lpIIR = designfilt('lowpassiir', ...
    'FilterOrder', N, ...
    'PassbandFrequency', Fc, ...
    'PassbandRipple', Rp, ...
    'StopbandAttenuation', Rst, ...
    'SampleRate', Fs, ...
    'DesignMethod', 'ellip');

% Convert the filter into second-order sections
[b, a] = tf(lpIIR);
[sos, g] = tf2sos(b, a);

% Fold the overall gain into the first stage
sos(1,1:3) = sos(1,1:3) * g;

disp('Coefficients scaled by 2^15:')
disp(round(sos * scale))
```

## High-pass Filter

![hpf.png](attachment:hpf.png)

Copy and paste the following MATLAB code:
```ts
Fc = 1000;      % Cutoff frequency in Hz (CHANGE THIS)

% Leave these values unchanged
Fs = 48100;     % Sample rate
N  = 4;         % Filter order
Rp = 1;         % Passband ripple in dB
Rst = 60;       % Stopband attenuation in dB
scale = 2^15;   % Fixed-point scaling

% Design the low-pass elliptic IIR filter
lpIIR = designfilt('highpassiir', ...
    'FilterOrder', N, ...
    'PassbandFrequency', Fc, ...
    'PassbandRipple', Rp, ...
    'StopbandAttenuation', Rst, ...
    'SampleRate', Fs, ...
    'DesignMethod', 'ellip');

% Convert the filter into second-order sections
[b, a] = tf(lpIIR);
[sos, g] = tf2sos(b, a);

% Fold the overall gain into the first stage
sos(1,1:3) = sos(1,1:3) * g;

disp('Coefficients scaled by 2^15:')
disp(round(sos * scale))
```

## Bandpass Filter

![bpf.png](attachment:bpf.png)

Copy and paste the following MATLAB code:
```ts
F_pass = [300 3400]; % Passband range in Hz (CHANGE THIS)

% Leave these values unchanged
Fs = 48100;       % Sample rate
N = 2;            % 2 SOS sections
Rp = 1;           % Passband ripple in dB
Rst = 60;         % Stopband attenuation in dB
scale = 2^15;     % Fixed-point scaling

% Normalize frequencies (0 to 1, where 1 is Nyquist)
Wn = F_pass / (Fs/2);

% Design using the 'ellip' function
[z, p, k] = ellip(N, Rp, Rst, Wn, 'bandpass');

% Convert to Second-Order Sections (SOS)
[sos, g] = zp2sos(z, p, k);

% Fold overall gain into the first section
sos(1,1:3) = sos(1,1:3) * g;

% Display Results
disp('Coefficients scaled by 2^15:')
disp(round(sos * scale))
```

## Bandstop Filter

![bsf.png](attachment:bsf.png)

Copy and paste the following MATLAB code:
```ts
F_pass = [300 3400]; % Passband range in Hz (CHANGE THIS)

% Leave these values unchanged
Fs = 48100;       % Sample rate
N = 2;            % 2 SOS sections
Rp = 1;           % Passband ripple in dB
Rst = 60;         % Stopband attenuation in dB
scale = 2^15;     % Fixed-point scaling

% Normalize frequencies (0 to 1, where 1 is Nyquist)
Wn = F_pass / (Fs/2);

% Design using the 'ellip' function
[z, p, k] = ellip(N, Rp, Rst, Wn, 'stop');

% Convert to Second-Order Sections (SOS)
[sos, g] = zp2sos(z, p, k);

% Fold overall gain into the first section
sos(1,1:3) = sos(1,1:3) * g;

% Display Results
disp('Coefficients scaled by 2^15:')
disp(round(sos * scale))
```

## What you will be doing 

Use these MATLAB scripts to generate coefficients for different filters. Try things out and see what happens.

---
|**Back:** [Filters](filters.ipynb)|
|----------------------------------|